# RL Post-Training — 0 : le reward model from scratch (Bradley-Terry)

**Série RL Post-Training** (`rlpt_*`) — cette série explore l'**alignement** de modèles de langage
sur un signal de préférence (RLHF, GRPO, DPO). Le pipeline complet d'InstructGPT comporte trois
étages : collecter des **comparaisons humaines**, en **entraîner un reward model**, puis optimiser
la politique contre ce reward model sous contrainte KL.

`RLPT-01` commence directement au troisième étage avec un reward model **jouet codé en dur**
(`rm(seq, prompt)` : `+1` si `a` en position 1 pour le prompt A, `e` en position 5 pour le prompt
B). Ce notebook est l'étage manquant, numéroté 0 parce qu'il le précède : **d'où vient le reward
quand personne ne peut l'écrire ?** Réponse : on l'apprend à partir de paires de préférences.

Le fil conducteur est le formalisme de **Bradley-Terry (1952)**, adapté aux préférences par
Christiano et al. (2017) puis Ouyang et al. (2022) :

$$P(y_w \succ y_l \mid x) = \sigma\big(r_\theta(x, y_w) - r_\theta(x, y_l)\big)$$

où $y_w$ (winning) et $y_l$ (losing) sont deux réponses au même prompt $x$, $\sigma$ la sigmoïde
et $r_\theta$ un réseau à tête scalaire. L'entraînement maximise la vraisemblance des choix
observés — exactement une régression logistique sur les **différences** de reward.

**Prérequis** : PyTorch de base (`RL-06`), le concept de reward (`RL-01`, `RL-05`). Dépendances CPU : TorchSharp-cpu 0.107.0 et MathNet.Numerics 5.0.0 : tout est from scratch, CPU, déterministe
(multi-seed `[0, 1, 7, 42]`, convention review C).

**Plan** : un monde synthétique avec un reward *vrai* connu (§2), un juge qui choisit entre deux
réponses selon Bradley-Terry (§3), un reward model entraîné par MLE (§4-6), et une évaluation
*honnête* — plafond de Bayes, calibration, identification par prompt (§7-10). Trois exercices clôturent
le notebook.

**Navigation** : [Index RL](../README.md) | [Version Python](RLPT-00-Reward-Model-Bradley-Terry-Python.ipynb) | [RLPT-00b](RLPT-00b-Biais-Dataset-Preferences-Python.ipynb)

**Durée estimée** : 90 minutes.

**Objectifs pédagogiques** : entraîner un modèle Bradley–Terry, évaluer accuracy, Brier et calibration face aux références de Bayes, puis identifier les limites de l’architecture et des comparaisons intra-prompt.

## 1. Le problème : noter est dur, comparer est facile

En RL classique, le reward est **donné** par l'environnement (gagner la partie, atteindre le but).
Pour aligner un modèle de langage, personne ne sait écrire $r(\text{« bonne réponse »})$. L'exploit
du RLHF (Christiano et al. 2017) est un déplacement : les humains ne savent pas **noter** une
réponse sur une échelle, mais savent très bien **comparer** deux réponses au même prompt.

Le contrat est donc :

| Ce qui existe | Ce qu'on veut |
|---|---|
| Des paires $(y_w, y_l)$ — « je préfère celle-ci à celle-là » | Une fonction $r_\theta(x, y)$ utilisable comme reward par PPO/GRPO (`RLPT-01`) |

Bradley-Terry transforme les préférences en probabilités : la paire fournit une observation bruitée dont la probabilité dépend de la différence des scores. C'est
une hypothèse de **rationalité du juge** — on verra précisément ce qu'elle vaut dans le monde
synthétique, où le vrai reward est connu et donc mesurable.

**Pourquoi un monde synthétique ?** Sur des données réelles, on ne peut pas mesurer l'erreur du
reward model (personne ne connaît $r^*$). Ici, $r^*$ est connu par construction : on peut calculer
la **référence de Bayes** (accuracy optimale en espérance conditionnelle, pour ces entrées et ce juge stochastique), le plancher de Brier, et la corrélation $\hat r$ vs $r^*$. C'est la
différence entre *évaluer* un reward model et *prouver* qu'il est honnête.

In [1]:
#r "nuget:TorchSharp-cpu,0.107.0"
#r "nuget:MathNet.Numerics,5.0.0"
using System;
using System.Linq;
using System.Collections.Generic;
using System.Globalization;
using System.Text;
using System.Security;
using TorchSharp;
using static TorchSharp.torch;
using MathNet.Numerics;
using MathNet.Numerics.Statistics;
using Microsoft.DotNet.Interactive.Formatting;

CultureInfo.DefaultThreadCurrentCulture = CultureInfo.InvariantCulture;
torch.set_num_threads(1);
torch.manual_seed(0);
int[] seeds =
{
    0,
    1,
    7,
    42
};
string[] tokens =
{
    "<pA>",
    "<pB>",
    "a",
    "b",
    "c",
    "d",
    "e",
    "f",
    "g",
    "h"
};
Console.WriteLine($".NET {Environment.Version}; TorchSharp {typeof(Tensor).Assembly.GetName().Version}; libtorch CPU 2.10.0; MathNet {typeof(Fit).Assembly.GetName().Version}; threads={torch.get_num_threads()}");
Console.WriteLine($"Vocabulaire {tokens.Length}, réponse 8 tokens, graines {string.Join(",", seeds)}.");

Installed Packages MathNet.Numerics, 5.0.0 TorchSharp-cpu, 0.107.0

Loading extensions from `/workspace/coursia-toolchains/nuget/skiasharp/2.88.6/interactive-extensions/dotnet/SkiaSharp.DotNet.Interactive.dll`

.NET 10.0.14; TorchSharp 0.107.0.0; libtorch CPU 2.10.0; MathNet 5.0.0.0; threads=1


Vocabulaire 10, réponse 8 tokens, graines 0,1,7,42.


## 2. Le monde synthétique : une oracle graduée

On reprend le monde de `RLPT-01` à l'identique — vocabulaire de 8 tokens de contenu `a`..`h`, deux
prompts `<pA>`/`<pB>`, réponses de 8 tokens — mais son oracle binaire (`0`/`1`) est remplacée par
une version **graduée**, prolongement naturel de la même structure :

- **poids par token** $w(t)$ : `a` porteur (+0.30), `e` porteur (+0.25), ... `h` bruit (−0.20) ;
- **bonus positionnel conditionné au prompt** : +0.80 si le prompt A reçoit `a` en position 1,
  +0.80 si le prompt B reçoit `e` en position 5 — exactement la structure de l'oracle de `RLPT-01`,
  en version continue.

Pourquoi graduer ? Parce que Bradley-Terry a besoin d'une **échelle** : avec un reward binaire,
toutes les paires non-ex-aequo ont la même différence $\Delta r^* = 1$, et le modèle de
préférence n'a rien à apprendre au-delà d'un classement à deux niveaux. Le reward gradué donne à
$\sigma(\Delta r)$ une plage complète de probabilités — c'est ce qui rend la *calibration*
(section 8) mesurable.

Le reward vrai d'une réponse est donc :

$$r^*(y \mid x) = \sum_{t \in y} w(t) + 0.8 \cdot \mathbb{1}[x = A,\ y_1 = a] + 0.8 \cdot \mathbb{1}[x = B,\ y_5 = e]$$

In [2]:
double[] poids =
{
    .30,
    .05,
    .10,
    .15,
    .25,
    -.10,
    -.15,
    -.20
};
double Oracle(int[] seq, int prompt)
{
    double r = seq.Sum(t => poids[t - 2]);
    if (prompt == 0 && seq[0] == 2)
        r += .8;
    if (prompt == 1 && seq[4] == 6)
        r += .8;
    return r;
}

string Texte(int[] s) => string.Join(" ", s.Select(i => tokens[i]));
int[][] exemples =
{
    new[]
    {
        2,
        9,
        9,
        9,
        9,
        9,
        9,
        9
    },
    Enumerable.Repeat(9, 8).ToArray(),
    new[]
    {
        3,
        3,
        3,
        3,
        6,
        3,
        3,
        3
    },
    new[]
    {
        2,
        4,
        5,
        2,
        3,
        6,
        8,
        4
    }
};
int[] promptsEx =
{
    0,
    0,
    1,
    0
};
for (int i = 0; i < 4; i++)
    Console.WriteLine($"prompt {tokens[promptsEx[i]]} | {Texte(exemples[i])} -> r*={Oracle(exemples[i], promptsEx[i]):F2}");
var ru = new Random(0);
var uniformes = Enumerable.Range(0, 2000).Select(_ => Oracle(Enumerable.Range(0, 8).Select(_ => ru.Next(2, 10)).ToArray(), ru.Next(2))).ToArray();
double mu = uniformes.Average();
Console.WriteLine($"2000 réponses uniformes : min={uniformes.Min():F2}, max={uniformes.Max():F2}, moyenne={mu:F2}, écart-type={Math.Sqrt(uniformes.Select(x => (x - mu) * (x - mu)).Average()):F2}");

prompt <pA> | a h h h h h h h -> r*=-0.30


prompt <pA> | h h h h h h h h -> r*=-1.60


prompt <pB> | b b b b e b b b -> r*=1.40


prompt <pA> | a c d a b e g c -> r*=1.90


2000 réponses uniformes : min=-1.25, max=2.60, moyenne=0.51, écart-type=0.60


## 3. Le juge : des préférences selon Bradley-Terry

Le **seul signal observable** n'est pas $r^*$ mais des choix binaires. Le juge (l'« humain »
simulé) reçoit deux réponses au même prompt et choisit :

$$P(y_a \succ y_b \mid x) = \sigma\big(\beta \,(r^*(y_a \mid x) - r^*(y_b \mid x))\big), \qquad \beta = 1$$

Deux choix de conception à noter :

1. **Paires à même prompt.** Comparer des réponses à des prompts différents n'a pas de sens (une
   réponse n'est bonne *que* relativement à sa question) — c'est aussi la contrainte du monde
   réel, où les plateformes de labeling présentent deux réponses au même prompt.
2. **Paires écartées si $|\Delta r^*| < 0.3$.** Un juge confronté à deux réponses quasi équivalentes
   répond à pile ou face ; ces paires ne contiennent presque pas de signal et diluent le dataset.
   Le monde réel fait de même : les comparaisons trop ambiguës sont bruitées pour l'annotateur.

L'**étiquette** est tirée de Bernoulli($P$) : le juge est *rationnel mais stochastique*. Ce détail
gouverne toute l'évaluation — même un reward model parfait commettra des « erreurs » sur les paires
où le juge s'est trompé. La référence de Bayes (section 7) quantifie cette limite en espérance, pas une borne dure sur une réalisation finie.

In [3]:
record Paire(int[] A, int[] B, int Prompt, int Y, double RA, double RB);
double Sigmoid(double z) => 1 / (1 + Math.Exp(-z));
List<Paire> Paires(int count, Random rng, double beta = 1, Func<double, double, double> juge = null)
{
    var data = new List<Paire>();
    while (data.Count < count)
    {
        int p = rng.Next(2);
        int[] a = Enumerable.Range(0, 8).Select(_ => rng.Next(2, 10)).ToArray(), b = Enumerable.Range(0, 8).Select(_ => rng.Next(2, 10)).ToArray();
        double ra = Oracle(a, p), rb = Oracle(b, p);
        if (Math.Abs(ra - rb) < .3)
            continue;
        double proba = juge is null ? Sigmoid(beta * (ra - rb)) : juge(ra - rb, beta);
        if (!double.IsFinite(proba) || proba < 0 || proba > 1)
            throw new Exception("Probabilité du juge invalide");
        int y = rng.NextDouble() < proba ? 1 : 0;
        data.Add(new Paire(a, b, p, y, ra, rb));
    }

    return data;
}

var rng = new Random(0);
var train = Paires(4000, rng);
var test = Paires(1000, rng);
var ex = train[3];
Console.WriteLine($"Train {train.Count} paires, test {test.Count}; filtre |Δ| ≥ 0,3; beta=1.");
Console.WriteLine($"Exemple prompt {tokens[ex.Prompt]}: a={Texte(ex.A)} r*={ex.RA:F2}; b={Texte(ex.B)} r*={ex.RB:F2}; juge={(ex.Y == 1 ? "a" : "b")}, P(a)={Sigmoid(ex.RA - ex.RB):F3}.");

Train 4000 paires, test 1000; filtre |Δ| ≥ 0,3; beta=1.


Exemple prompt <pB>: a=g h c e f c c g r*=-0.05; b=b b a e f f h b r*=0.30; juge=b, P(a)=0.413.


### Statistiques du juge

Les quantiles et taux ci-dessous résument les paires générées.

In [4]:
var dTrain = train.Select(x => x.RA - x.RB).OrderBy(x => x).ToArray();
double Quantile(double[] a, double q)
{
    double p = (a.Length - 1) * q;
    int i = (int)p;
    return a[i] + (a[Math.Min(i + 1, a.Length - 1)] - a[i]) * (p - i);
}

foreach (int q in new[]
{
    0,
    10,
    25,
    50,
    75,
    90,
    100
}

)
    Console.WriteLine($"Δr* q{q, 3} : {Quantile(dTrain, q / 100d):F2}");
double oracleAcc = test.Average(x => (x.RA > x.RB) == (x.Y == 1) ? 1d : 0d);
double bayes = test.Average(x =>
{
    double p = Sigmoid(x.RA - x.RB);
    return Math.Max(p, 1 - p);
});
Console.WriteLine($"Accord juge réalisé={oracleAcc:F3}; Bayes en espérance={bayes:F3}; labels 1 train={train.Average(x => (double)x.Y):F3}.");

Δr* q  0 : -2.90


Δr* q 10 : -1.25


Δr* q 25 : -0.80


Δr* q 50 : -0.35


Δr* q 75 : 0.75


Δr* q 90 : 1.20


Δr* q100 : 3.10


Accord juge réalisé=0.683; Bayes en espérance=0.696; labels 1 train=0.491.


**Lecture des statistiques.** Le tableau mesure le taux d'accord réalisé du juge, la référence de Bayes en espérance conditionnelle et les quantiles. Une accuracy réalisée sur un échantillon fini peut dépasser cette référence par fluctuation.

## 4. Encodage : une paire = deux séquences `[prompt] + réponse`

Le reward model de la littérature note la **paire** $(x, y)$ — il est conditionné au prompt, comme
l'oracle. On encode chaque réponse en **9 tokens** : le token de prompt en position 0, la réponse
en positions 1..8, exactement la mise en page de la politique de `RLPT-01` (`x = [prompt, y_1..y_8]`).

Une paire devient un tenseur `(B, 2, 9)` : deux « bras » que le modèle note indépendamment. La
perte BT ne voit jamais les scores absolus séparément — seulement leur **différence** — c'est le
cœur de la section 10.

In [5]:
long[] Encode(int prompt, int[] seq) => new[]
{
    (long)prompt
}.Concat(seq.Select(x => (long)x)).ToArray();
foreach (var(seq, j)in new[]
{
    ex.A,
    ex.B
}.Select((s, j) => (s, j)))
    Console.WriteLine($"bras {(j == 0 ? "a" : "b")} : [{tokens[ex.Prompt]}] {Texte(seq)}; longueur encodée={Encode(ex.Prompt, seq).Length}");
Console.WriteLine($"Tenseurs logiques : train=({train.Count},2,9), test=({test.Count},2,9).");

bras a : [<pB>] g h c e f c c g; longueur encodée=9


bras b : [<pB>] b b a e f f h b; longueur encodée=9


Tenseurs logiques : train=(4000,2,9), test=(1000,2,9).


## 5. Le reward model : embedding + position + pooling + tête scalaire

La séquence de neuf tokens passe par `Embedding(10,64)` et `Embedding(9,64)`, puis moyenne et MLP 64→64 ReLU→1 : **5 441 paramètres**. La somme des embeddings de position est constante à longueur fixe. Le pooling est donc invariant aux permutations du contenu et ne peut pas apprendre le bonus dépendant de la position, même si les positions sont encodées. La cellule vérifie ce contre-exemple.

In [6]:
class RewardModel : IDisposable
{
    public TorchSharp.Modules.Embedding emb = torch.nn.Embedding(10, 64);
    public TorchSharp.Modules.Embedding pos = torch.nn.Embedding(9, 64);
    public torch.nn.Module<Tensor, Tensor> mlp = torch.nn.Sequential(torch.nn.Linear(64, 64), torch.nn.ReLU(), torch.nn.Linear(64, 1));
    public Tensor Forward(Tensor x)
    {
        var h = emb.forward(x) + pos.forward(torch.arange(9, dtype: ScalarType.Int64));
        return mlp.forward(h.mean(new long[] { 1 })).squeeze(1);
    }

    public IEnumerable<TorchSharp.Modules.Parameter> Parameters() => emb.parameters().Concat(pos.parameters()).Concat(mlp.parameters());
    public void Dispose()
    {
        emb.Dispose();
        pos.Dispose();
        mlp.Dispose();
    }
}

var initial = new RewardModel();
long parametres = initial.Parameters().Sum(x => x.numel());
if (parametres != 5441)
    throw new Exception("Nombre de paramètres");
Console.WriteLine($"Embedding + position + moyenne + MLP : {parametres} paramètres.");
using (var scope = torch.NewDisposeScope())
{
    var enc = torch.tensor(exemples.SelectMany((s, i) => Encode(promptsEx[i], s)).ToArray()).reshape(4, 9);
    var valeurs = initial.Forward(enc).data<float>().ToArray();
    for (int i = 0; i < 4; i++)
        Console.WriteLine($"{tokens[promptsEx[i]]} r*={Oracle(exemples[i], promptsEx[i]):F2}, score initial={valeurs[i]:F3}");
    var a = new[]
    {
        2,
        9,
        9,
        9,
        9,
        9,
        9,
        9
    };
    var b = new[]
    {
        9,
        2,
        9,
        9,
        9,
        9,
        9,
        9
    };
    var x = torch.tensor(Encode(0, a).Concat(Encode(0, b)).ToArray()).reshape(2, 9);
    var y = initial.Forward(x).data<float>().ToArray();
    double delta = Oracle(a, 0) - Oracle(b, 0);
    if (Math.Abs(delta - .8) > 1e-12 || Math.Abs(y[0] - y[1]) > 1e-6)
        throw new Exception("Permutation");
    Console.WriteLine($"Permutation : delta oracle={delta:F3}, delta modèle={y[0] - y[1]:E3}.");
}

initial.Dispose();

Embedding + position + moyenne + MLP : 5441 paramètres.


<pA> r*=-0.30, score initial=-0.132


<pA> r*=-1.60, score initial=-0.116


<pB> r*=1.40, score initial=-0.055


<pA> r*=1.90, score initial=-0.102


Permutation : delta oracle=0.800, delta modèle=-2.980E-008.



warning CS1701: Assuming assembly reference 'System.Runtime.Numerics, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' used by 'TorchSharp' matches identity 'System.Runtime.Numerics, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' of 'System.Runtime.Numerics', you may need to supply runtime policy

warning CS1701: Assuming assembly reference 'System.Runtime.Numerics, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' used by 'TorchSharp' matches identity 'System.Runtime.Numerics, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' of 'System.Runtime.Numerics', you may need to supply runtime policy



## 6. Entraînement : vraisemblance de Bradley–Terry

La BCE sur le logit gagnant moins perdant apprend les probabilités conditionnelles. Elle est un score propre : elle ne crée pas par elle-même une surconfiance structurelle. Le label 1 désigne le bras 0, donc `winner = 1-y`, `loser = y`. Adam, 60 époques et batch 256 sont conservés.

In [7]:
(RewardModel model, List<double> pertes) Entraine(List<Paire> data, int seed, bool verbose = false, Func<Tensor, Tensor, double, Tensor> penalty = null, double lam = 0)
{
    torch.manual_seed(seed);
    var r = new Random(seed);
    var model = new RewardModel();
    using var optim = torch.optim.Adam(model.Parameters(), lr: .01);
    var hist = new List<double>();
    int n = data.Count;
    for (int ep = 0; ep < 60; ep++)
    {
        int[] ordre = Enumerable.Range(0, n).OrderBy(_ => r.Next()).ToArray();
        double total = 0;
        for (int start = 0; start < n; start += 256)
        {
            using var scope = torch.NewDisposeScope();
            var lot = ordre.Skip(start).Take(256).ToArray();
            var win = new List<long>();
            var lose = new List<long>();
            foreach (int j in lot)
            {
                var x = data[j];
                win.AddRange(Encode(x.Prompt, x.Y == 1 ? x.A : x.B));
                lose.AddRange(Encode(x.Prompt, x.Y == 1 ? x.B : x.A));
            }

            var a = torch.tensor(win.ToArray()).reshape(lot.Length, 9);
            var b = torch.tensor(lose.ToArray()).reshape(lot.Length, 9);
            var rw = model.Forward(a); var rl = model.Forward(b);
            var logits = rw - rl;
            var loss = torch.nn.functional.softplus(-logits).mean();
            if (penalty is not null) loss = loss + penalty(rw, rl, lam);
            optim.zero_grad();
            loss.backward();
            optim.step();
            total += loss.item<float>() * lot.Length;
        }

        hist.Add(total / n);
        if (verbose && (ep % 15 == 0 || ep == 59))
            Console.WriteLine($"époque {ep, 2} : NLL={hist[^1]:F4}");
    }

    return (model, hist);
}

var resultat = Entraine(train, 0, true);
var model = resultat.model;
var hist = resultat.pertes;
Console.WriteLine($"Entraînement BT : {hist.Count} époques, NLL {hist[0]:F4} → {hist[^1]:F4}, batch 256, Adam 0,01.");

époque  0 : NLL=0.6368


époque 15 : NLL=0.6100


époque 30 : NLL=0.6056


époque 45 : NLL=0.5991


époque 59 : NLL=0.5929


Entraînement BT : 60 époques, NLL 0.6368 → 0.5929, batch 256, Adam 0,01.



warning CS1701: Assuming assembly reference 'System.Runtime.Numerics, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' used by 'TorchSharp' matches identity 'System.Runtime.Numerics, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' of 'System.Runtime.Numerics', you may need to supply runtime policy

warning CS1701: Assuming assembly reference 'System.Runtime.Numerics, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' used by 'TorchSharp' matches identity 'System.Runtime.Numerics, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' of 'System.Runtime.Numerics', you may need to supply runtime policy



## 7. Évaluation : références de Bayes

Pour $p=\sigma(\Delta r^*)$, l'accuracy optimale conditionnelle **en espérance** est $E[\max(p,1-p)]$ et le Brier Bayes est $E[p(1-p)]$. Une réalisation finie peut dépasser la première. La NLL oracle réalisée et l'entropie attendue sont deux quantités distinctes ; l'écart train/test est un diagnostic, non une preuve de surapprentissage.

In [8]:
(double[] scores, double[] prob, double acc, double brier, double nll) Predire(RewardModel m, List<Paire> data)
{
    using var scope = torch.NewDisposeScope();
    using var ng = torch.no_grad();
    long[] x = data.SelectMany(p => Encode(p.Prompt, p.A).Concat(Encode(p.Prompt, p.B))).ToArray();
    var t = torch.tensor(x).reshape(data.Count * 2, 9);
    var f = m.Forward(t).data<float>().ToArray();
    var scores = f.Select(x => (double)x).ToArray();
    var pp = Enumerable.Range(0, data.Count).Select(i => Sigmoid(scores[2 * i] - scores[2 * i + 1])).ToArray();
    double acc = Enumerable.Range(0, data.Count).Average(i => (pp[i] > .5) == (data[i].Y == 1) ? 1d : 0d);
    double br = Enumerable.Range(0, data.Count).Average(i => Math.Pow(pp[i] - data[i].Y, 2));
    double nll = Enumerable.Range(0, data.Count).Average(i => -(data[i].Y * Math.Log(pp[i] + 1e-12) + (1 - data[i].Y) * Math.Log(1 - pp[i] + 1e-12)));
    return (scores, pp, acc, br, nll);
}

var pred = Predire(model, test);
var predTrain = Predire(model, train);
double[] vrai = test.SelectMany(p => new[] { p.RA, p.RB }).ToArray();
double[] ptrue = test.Select(p => Sigmoid(p.RA - p.RB)).ToArray();
double brierBayes = ptrue.Average(p => p * (1 - p));
double entropie = ptrue.Average(p => -(p * Math.Log(p) + (1 - p) * Math.Log(1 - p)));
double nllOracle = Enumerable.Range(0, test.Count).Average(i => -(test[i].Y * Math.Log(ptrue[i]) + (1 - test[i].Y) * Math.Log(1 - ptrue[i])));
double rho = Correlation.Spearman(pred.scores, vrai);
Console.WriteLine($"Accuracy test={pred.acc:F3}, train={predTrain.acc:F3}, Bayes en espérance={bayes:F3}");
Console.WriteLine($"Brier test={pred.brier:F4}, référence Bayes={brierBayes:F4}; Spearman global={rho:F3}");
Console.WriteLine($"NLL train={hist[^1]:F4}, modèle test={pred.nll:F4}, oracle réalisé={nllOracle:F4}, entropie attendue={entropie:F4}");

Accuracy test=0.627, train=0.699, Bayes en espérance=0.696


Brier test=0.2291, référence Bayes=0.2038; Spearman global=0.788


NLL train=0.5929, modèle test=0.6525, oracle réalisé=0.6096, entropie attendue=0.5945



warning CS1701: Assuming assembly reference 'System.Runtime.Numerics, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' used by 'TorchSharp' matches identity 'System.Runtime.Numerics, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' of 'System.Runtime.Numerics', you may need to supply runtime policy



**Lecture du tableau.** L'accuracy réalisée se compare à la référence en espérance, et le Brier du modèle à $E[p(1-p)]$. Le tableau donne aussi NLL train, NLL test, NLL oracle réalisée et entropie attendue : leur comparaison renseigne sans prouver à elle seule une cause. Spearman global agrège deux prompts dont les offsets ne sont pas identifiés.

Sur ce run, accuracy test **0,627** contre référence attendue **0,696** ; Brier **0,2291** contre **0,2038**. NLL modèle test **0,6525** et entropie attendue **0,5945** : des écarts à interpréter avec la capacité et le bruit.

## 8. Calibration : diagramme de fiabilité

Chaque point est un décile non vide : probabilité moyenne prédite contre fréquence observée. La diagonale $y=x$ représente la calibration parfaite ; l'ECE pondère les écarts par les effectifs. Le bruit fini, la capacité et l'optimisation peuvent contribuer aux écarts.

In [9]:
record RewardSvg(string Content);
Formatter.Register<RewardSvg>((v, w) => w.Write(v.Content), "text/html");
string F(double v) => v.ToString("0.###", CultureInfo.InvariantCulture);
string E(string v) => SecurityElement.Escape(v) ?? "";
string Scatter(string titre, string xlab, string ylab, (string nom, string couleur, double[] x, double[] y)[] series, double xmin, double xmax, double ymin, double ymax, bool diagonale = false, string[] etiquettes = null)
{
    double X(double v) => 105 + 550 * (v - xmin) / (xmax - xmin);
    double Y(double v) => 325 - 250 * (v - ymin) / (ymax - ymin);
    var s = new StringBuilder($"<svg xmlns='http://www.w3.org/2000/svg' width='760' height='420' viewBox='0 0 760 420'><rect width='100%' height='100%' fill='white'/><text x='380' y='23' text-anchor='middle'>{E(titre)}</text><path d='M105 75 V325 H655' stroke='black' fill='none'/>");
    for (int i = 0; i <= 4; i++)
    {
        double v = xmin + (xmax - xmin) * i / 4, w = ymin + (ymax - ymin) * i / 4;
        s.Append($"<path d='M105 {F(Y(w))} H655' stroke='#ddd'/><text x='{F(X(v))}' y='345' text-anchor='middle'>{v:F2}</text><text x='98' y='{F(Y(w) + 4)}' text-anchor='end'>{w:F2}</text>");
    }

    if (diagonale)
        s.Append($"<path d='M{F(X(0))} {F(Y(0))} L{F(X(1))} {F(Y(1))}' stroke='black' stroke-dasharray='5,4'/>");
    for (int j = 0; j < series.Length; j++)
    {
        var z = series[j];
        if (z.nom == "fit affine")
            s.Append($"<path d='M{F(X(z.x[0]))} {F(Y(z.y[0]))} L{F(X(z.x[1]))} {F(Y(z.y[1]))}' stroke='{z.couleur}' fill='none'/>");
        else
            for (int i = 0; i < z.x.Length; i++)
            {
                s.Append($"<circle cx='{F(X(z.x[i]))}' cy='{F(Y(z.y[i]))}' r='2.5' fill='{z.couleur}' fill-opacity='0.5'/>");
                if (etiquettes != null && j == 0)
                    s.Append($"<text x='{F(X(z.x[i]) + 4)}' y='{F(Y(z.y[i]) - 4)}' font-size='10'>{E(etiquettes[i])}</text>");
            }

        s.Append($"<text x='{115 + j * 180}' y='52' fill='{z.couleur}'>{E(z.nom)}</text>");
    }

    s.Append($"<text x='380' y='392' text-anchor='middle'>{E(xlab)}</text><text x='15' y='200' transform='rotate(-90 15 200)' text-anchor='middle'>{E(ylab)}</text></svg>");
    return s.ToString();
}

var bins = Enumerable.Range(0, 10).Select(k => Enumerable.Range(0, test.Count).Where(i => Math.Min((int)(pred.prob[i] * 10), 9) == k).ToArray()).ToArray();
var nonVides = Enumerable.Range(0, 10).Where(k => bins[k].Length > 0).ToArray();
double[] conf = nonVides.Select(k => bins[k].Average(i => pred.prob[i])).ToArray(), obs = nonVides.Select(k => bins[k].Average(i => (double)test[i].Y)).ToArray();
double ece = 0;
for (int j = 0; j < nonVides.Length; j++)
{
    int k = nonVides[j];
    ece += bins[k].Length * Math.Abs(conf[j] - obs[j]) / test.Count;
    Console.WriteLine($"décile {k}: n={bins[k].Length}, prédit={conf[j]:F3}, observé={obs[j]:F3}");
}

double confiance = pred.prob.Average(p => Math.Max(p, 1 - p));
Console.WriteLine($"ECE={ece:F4}; confiance moyenne={confiance:F3}, accuracy={pred.acc:F3}.");
display(new RewardSvg(Scatter("Fiabilité : déciles non vides", "Probabilité moyenne prédite", "Fréquence observée", new[] { ("déciles", "#0645ad", conf, obs) }, 0, 1, 0, 1, true, nonVides.Select(k => k.ToString()).ToArray())));

décile 0: n=20, prédit=0.081, observé=0.300


décile 1: n=84, prédit=0.161, observé=0.262


décile 2: n=135, prédit=0.252, observé=0.356


décile 3: n=123, prédit=0.349, observé=0.374


décile 4: n=142, prédit=0.451, observé=0.472


décile 5: n=127, prédit=0.550, observé=0.488


... (2 outputs left out)

décile 8: n=86, prédit=0.841, observé=0.779


décile 9: n=14, prédit=0.930, observé=1.000


ECE=0.0641; confiance moyenne=0.689, accuracy=0.627.


Fiabilité : déciles non vides 0.00 0.00 0.25 0.25 0.50 0.50 0.75 0.75 1.00 1.00 0 1 2 3 4 5 6 7 8 9 déciles Probabilité moyenne prédite Fréquence observée

**Lecture du diagramme.** Les effectifs et valeurs sont imprimés pour chaque décile non vide. L'ECE, la confiance et l'accuracy décrivent cet échantillon ; ils ne prouvent pas une surconfiance structurelle de la perte BT, qui est un score propre.

### Exercice 1 : régulariser le score

Complétez la fonction de pénalité tensorielle $\lambda E[r_w^2+r_l^2]$. Conservez le graphe de gradients : pas de conversion en nombre ni de détachement. Le harnais vérifie valeur et gradients sur plusieurs fixtures, puis entraîne lui-même trois modèles avec Adam, 60 époques, mêmes données et graine pour $\lambda=0, 0{,}01, 0{,}1$. Comparez Brier, ECE et accuracy ; la pénalité peut changer l'échelle. Ces fixtures testent le contrat sur des exemples, pas une preuve pour toutes les entrées.

In [10]:
Func<Tensor, Tensor, double, Tensor> penaliteScores = null; // TODO : scalaire différentiable
void VerifierPenalite(Func<Tensor, Tensor, double, Tensor> fn) {
    foreach (var (av,bv,lam) in new[] {
        (new float[]{1,2},new float[]{3,4},.1),
        (new float[]{-2,0,2},new float[]{1,-1,3},.01),
        (new float[]{3,4},new float[]{2,1},0d) }) {
        using var scope=torch.NewDisposeScope();
        var a=torch.tensor(av, requires_grad:true); var b=torch.tensor(bv, requires_grad:true);
        var v=fn(a,b,lam);
        if(v is null || v.numel()!=1 || !double.IsFinite(v.item<float>()) || !v.requires_grad) throw new Exception("Pénalité scalaire différentiable requise");
        double expected=lam*av.Zip(bv,(x,y)=>(double)x*x+(double)y*y).Average();
        if(Math.Abs(v.item<float>()-expected)>1e-6) throw new Exception("Valeur incorrecte");
        v.backward();
        if(a.grad is null || b.grad is null) throw new Exception("Gradient absent");
        var ga=a.grad.data<float>().ToArray(); var gb=b.grad.data<float>().ToArray();
        for(int j=0;j<av.Length;j++) if(Math.Abs(ga[j]-2*lam*av[j]/av.Length)>1e-6 || Math.Abs(gb[j]-2*lam*bv[j]/bv.Length)>1e-6) throw new Exception("Gradient incorrect");
    }
}
if(penaliteScores is null) Console.WriteLine("Exercice 1 à compléter : pénalité tensorielle.");
else {
    VerifierPenalite(penaliteScores);
    foreach(double lam in new[]{0d,.01,.1}) {
        var run=Entraine(train,0,penalty:penaliteScores,lam:lam);
        if(run.pertes.Count!=60 || run.pertes.Any(x=>!double.IsFinite(x))) throw new Exception("Entraînement invalide");
        var pr=Predire(run.model,test); double e=0;
        for(int k=0;k<10;k++){int[] ids=Enumerable.Range(0,test.Count).Where(i=>Math.Min((int)(pr.prob[i]*10),9)==k).ToArray();if(ids.Length>0)e+=ids.Length*Math.Abs(ids.Average(i=>pr.prob[i])-ids.Average(i=>(double)test[i].Y))/test.Count;}
        Console.WriteLine($"lambda={lam}: 60 époques Adam, accuracy={pr.acc:F3}, Brier={pr.brier:F4}, ECE={e:F4}"); run.model.Dispose();
    }
}

Exercice 1 à compléter : pénalité tensorielle.



warning CS1701: Assuming assembly reference 'System.Runtime.Numerics, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' used by 'TorchSharp' matches identity 'System.Runtime.Numerics, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' of 'System.Runtime.Numerics', you may need to supply runtime policy

warning CS1701: Assuming assembly reference 'System.Runtime.Numerics, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' used by 'TorchSharp' matches identity 'System.Runtime.Numerics, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' of 'System.Runtime.Numerics', you may need to supply runtime policy



## 9. Score et ajustement affine

Le nuage des 2000 scores est coloré par prompt. Le fit affine et Spearman résument la relation globale, sans identifier les offsets propres aux prompts ni prouver une calibration. À température fixée, modifier l'échelle des différences modifie généralement les probabilités.

In [11]:
var fit = Fit.Line(vrai, pred.scores);
double intercept = fit.Item1, pente = fit.Item2;
double mean = pred.scores.Average();
double ss = pred.scores.Sum(v => (v - mean) * (v - mean));
double err = Enumerable.Range(0, vrai.Length).Sum(i => Math.Pow(pred.scores[i] - (intercept + pente * vrai[i]), 2));
double r2 = 1 - err / ss;
Console.WriteLine($"Fit r̂ = {pente:F3} r* + {intercept:F3}; R²={r2:F3}; variance inexpliquée={1 - r2:F3}; Spearman global={rho:F3}.");
var series = Enumerable.Range(0, 2).Select(p =>
{
    var indices = Enumerable.Range(0, test.Count).Where(i => test[i].Prompt == p).SelectMany(i => new[] { 2 * i, 2 * i + 1 }).ToArray();
    return (p == 0 ? "prompt A" : "prompt B", p == 0 ? "#0645ad" : "#c35100", indices.Select(i => vrai[i]).ToArray(), indices.Select(i => pred.scores[i]).ToArray());
}).ToArray();
double xmin = vrai.Min() - .1, xmax = vrai.Max() + .1;
double[] lineX =
{
    xmin,
    xmax
};
double[] lineY = lineX.Select(x => intercept + pente * x).ToArray();
double ymin = Math.Min(pred.scores.Min(), lineY.Min()) - .1, ymax = Math.Max(pred.scores.Max(), lineY.Max()) + .1;
var avecFit = series.Concat(new[] { ("fit affine", "#a00000", lineX, lineY) }).ToArray();
display(new RewardSvg(Scatter("Scores estimés et oracle", "Reward vrai r*", "Reward estimé", avecFit, xmin, xmax, ymin, ymax)));

Fit r̂ = 0.879 r* + -0.921; R²=0.611; variance inexpliquée=0.389; Spearman global=0.788.


Scores estimés et oracle -1.20 -2.69 -0.15 -1.38 0.90 -0.06 1.95 1.25 3.00 2.57 prompt A prompt B fit affine Reward vrai r* Reward estimé

**Lecture du nuage.** La pente, l'interception et $R^2$ sont imprimés avec le graphique. $1-R^2$ représente la fraction de variance non expliquée par ce fit, sans être nécessairement du bruit. Les offsets entre prompts ne sont pas fixés par les comparaisons intra-prompt.

## 10. Identification : offsets par prompt

Une constante différente par prompt disparaît des différences intra-prompt. En revanche, multiplier les scores change les probabilités lorsque $\beta$ est fixé. Les deux contrôles suivants calculent effectivement ces écarts.

In [12]:
double ecartOffset = 0, ecartEchelle = 0;
for (int i = 0; i < test.Count; i++)
{
    double c = test[i].Prompt == 0 ? 5 : -3;
    double a = pred.scores[2 * i], b = pred.scores[2 * i + 1];
    ecartOffset = Math.Max(ecartOffset, Math.Abs(Sigmoid((a + c) - (b + c)) - pred.prob[i]));
    ecartEchelle = Math.Max(ecartEchelle, Math.Abs(Sigmoid(2 * (a - b)) - pred.prob[i]));
}

if (ecartOffset > 1e-10 || ecartEchelle < 1e-6)
    throw new Exception("Contrôle de jauge");
Console.WriteLine($"Offsets par prompt +5/-3 : max |Δp|={ecartOffset:E3}; échelle ×2 : max |Δp|={ecartEchelle:F3}.");

Offsets par prompt +5/-3 : max |Δp|=0.000E+000; échelle ×2 : max |Δp|=0.150.


### Exercice 2 : un juge plus bruité

Complétez `probabiliteJuge`, puis régénérez les paires train/test avec $\beta=0,5$, réentraînez et comparez accuracy, référence Bayes, NLL et pente. Le bruit d'échantillonnage et la limite de capacité empêchent de garantir un ordre ou une pente observée.

Étapes : calculez la sigmoïde à partir de `beta * delta` ; vérifiez le cas nul et les signes opposés ; régénérez ensuite train/test et comparez les métriques au même protocole.

In [13]:
Func<double, double, double?> probabiliteJuge = null; // TODO : sigmoid(beta × delta)
if (probabiliteJuge is null)
    Console.WriteLine("Exercice 2 à compléter : juge beta=0,5 et entraînement.");
else
{
    foreach (var(d, b)in new[]
    {
        (0d, .5),
        (1d, .5),
        (-2d, .5),
        (2d, 1d)
    }

    )
    {
        double? p = probabiliteJuge(d, b);
        if (p is null || !double.IsFinite(p.Value) || Math.Abs(p.Value - Sigmoid(b * d)) > 1e-10)
            throw new Exception("Probabilité incorrecte");
    }

    int appels = 0;
    double Juge(double d, double b)
    {
        appels++;
        return probabiliteJuge(d, b) ?? double.NaN;
    }

    var rr = new Random(0);
    var tr = Paires(4000, rr, .5, Juge);
    var te = Paires(1000, rr, .5, Juge);
    if (appels != 5000)
        throw new Exception("Juge non appelé");
    var run = Entraine(tr, 0);
    var pe = Predire(run.model, te);
    var ps = te.Select(x => Sigmoid(.5 * (x.RA - x.RB))).ToArray();
    var truth = te.SelectMany(x => new[] { x.RA, x.RB }).ToArray();
    var fitBeta = Fit.Line(truth, pe.scores);
    Console.WriteLine($"{appels} appels juge; accuracy={pe.acc:F3}, Bayes attendu={ps.Average(x => Math.Max(x, 1 - x)):F3}, pente={fitBeta.Item2:F3}, NLL test={pe.nll:F4}, Brier={pe.brier:F4}");
    run.model.Dispose();
}

Exercice 2 à compléter : juge beta=0,5 et entraînement.


## 11. Multi-seed : stabilité de la boucle

Quatre runs complets pour les graines 0, 1, 7 et 42 régénèrent données et juge, et réinitialisent les paramètres et l’ordre des lots. La dispersion est celle du pipeline de bout en bout, pas seulement celle de l'initialisation.

In [14]:
var runs = new List<(int seed, double accTrain, double acc, double bayes, double brier, double baseBrier, double rho, double loss)>();
foreach (int seed in seeds)
{
    var rr = new Random(seed);
    var tr = Paires(4000, rr);
    var te = Paires(1000, rr);
    var run = Entraine(tr, seed);
    var pe = Predire(run.model, te);
    var pt = Predire(run.model, tr);
    double[] ps = te.Select(x => Sigmoid(x.RA - x.RB)).ToArray();
    double[] truth = te.SelectMany(x => new[] { x.RA, x.RB }).ToArray();
    runs.Add((seed, pt.acc, pe.acc, ps.Average(x => Math.Max(x, 1 - x)), pe.brier, ps.Average(x => x * (1 - x)), Correlation.Spearman(pe.scores, truth), run.pertes[^1]));
    run.model.Dispose();
}

Console.WriteLine($"{"seed", 5} {"train", 7} {"test", 7} {"Bayes", 7} {"Brier", 8} {"réf.", 8} {"rho", 7} {"NLL", 7}");
foreach (var x in runs)
    Console.WriteLine($"{x.seed, 5} {x.accTrain, 7:F3} {x.acc, 7:F3} {x.bayes, 7:F3} {x.brier, 8:F4} {x.baseBrier, 8:F4} {x.rho, 7:F3} {x.loss, 7:F3}");
double moyenne = runs.Average(x => x.acc), sd = Math.Sqrt(runs.Average(x => (x.acc - moyenne) * (x.acc - moyenne)));
Console.WriteLine($"Quatre runs complets : accuracy moyenne={moyenne:F3} ± {sd:F3}; Brier moyen={runs.Average(x => x.brier):F4}, référence={runs.Average(x => x.baseBrier):F4}.");

 seed   train    test   Bayes    Brier     réf.     rho     NLL


    0   0.699   0.627   0.696   0.2291   0.2038   0.788   0.593


    1   0.700   0.703   0.697   0.2025   0.2041   0.780   0.591


    7   0.694   0.650   0.696   0.2220   0.2042   0.769   0.597


   42   0.695   0.648   0.692   0.2197   0.2055   0.778   0.583


Quatre runs complets : accuracy moyenne=0.657 ± 0.028; Brier moyen=0.2183, référence=0.2044.


**Lecture multi-seed.** La table affiche pour chaque graine accuracy train/test, référence de Bayes attendue, Brier et sa référence $E[p(1-p)]$, Spearman et perte. Les moyennes et écarts-types résument quatre expériences complètes ; elles ne démontrent pas une propriété structurelle de la BCE.

Les quatre runs donnent une accuracy moyenne **0,657 ± 0,028** et un Brier moyen **0,2183**, référence **0,2044**.

### Exercice 3 : budget d'étiquettes

Complétez `indicesProches`, puis générez un réservoir de **8000** paires et comparez **1500** paires aléatoires à **1500** proches sur le même test. Entraînez deux modèles et mesurez accuracy et Spearman, sans supposer un gagnant.

Étapes : triez les indices par écart absolu avec départage stable ; sélectionnez le budget sans doublon ; entraînez et comparez les deux politiques sur le test commun.

In [15]:
Func<double[], int, int[]> indicesProches = null; // TODO : indices des plus petits écarts absolus, ordre stable
if (indicesProches is null)
    Console.WriteLine("Exercice 3 à compléter : réservoir 8000 et comparaison de 1500 paires.");
else
{
    foreach (var(d, k, attendu)in new[]
    {
        (new[] { .9, .4, .7 }, 2, new[] { 1, 2 }),
        (new[] { -.8, .3, -.4, .9 }, 3, new[] { 1, 2, 0 })
    }

    )
    {
        var v = indicesProches(d, k);
        if (v is null || !v.SequenceEqual(attendu))
            throw new Exception("Sélection incorrecte");
    }

    var rr = new Random(0);
    var pool = Paires(8000, rr);
    double[] deltas = pool.Select(x => x.RA - x.RB).ToArray();
    var proche = indicesProches(deltas, 1500);
    var oracle = Enumerable.Range(0, 8000).OrderBy(i => Math.Abs(deltas[i])).ThenBy(i => i).Take(1500).ToArray();
    if (proche is null || !proche.SequenceEqual(oracle))
        throw new Exception("Sélection de 1500 incorrecte");
    var hasard = Enumerable.Range(0, 8000).OrderBy(_ => rr.Next()).Take(1500).ToArray();
    foreach (var(nom, ids)in new[]
    {
        ("aléatoire", hasard),
        ("proche", proche)
    }

    )
    {
        var run = Entraine(ids.Select(i => pool[i]).ToList(), 0);
        var pp = Predire(run.model, test);
        double r = Correlation.Spearman(pp.scores, vrai);
        Console.WriteLine($"{nom}: 1500 paires, accuracy={pp.acc:F3}, Spearman={r:F3}, Brier={pp.brier:F4}");
        run.model.Dispose();
    }
}

Exercice 3 à compléter : réservoir 8000 et comparaison de 1500 paires.


## À retenir

| Question | Résultat |
|---|---|
| Signal de préférence | Bradley–Terry compare les deux scores ; le bras gagnant vaut `1-y`. |
| Références | Accuracy Bayes en espérance et Brier $E[p(1-p)]$. |
| Limite architecturale | Le pooling moyen ignore l'ordre et ne peut apprendre le bonus positionnel. |
| Identification | Un offset par prompt est invisible, une échelle modifiée ne l'est pas à beta fixé. |
| Dispersion | Quatre graines régénèrent les données et relancent l’entraînement complet. |

## Exercices

1. [Régulariser le score](#Exercice-1-:-régulariser-le-score) : définir une pénalité différentiable et comparer trois intensités sur les mêmes données.
2. [Un juge plus bruité](#Exercice-2-:-un-juge-plus-bruité) : modifier la température du juge puis réévaluer les références et le modèle.
3. [Budget d'étiquettes](#Exercice-3-:-budget-d'étiquettes) : sélectionner des paires proches et comparer leur intérêt à un échantillon aléatoire.

## Conclusion

La vraie boucle optimise un modèle de reward sur des préférences bruitées. Les scores, références en espérance et graphiques permettent de distinguer capacité, calibration et dispersion ; la preuve de permutation expose une limite structurelle de cette architecture. La suite branche un reward sur une politique.

## Vérifiez votre compréhension

1. Pourquoi le Brier Bayes vaut-il $p(1-p)$ ?
2. Pourquoi le modèle ne sait-il pas distinguer deux permutations à prompt fixé ?
3. Quel changement de score préserve les probabilités BT ?

<details><summary>Réponses</summary>

1. C'est l'espérance de $(p-y)^2$ pour $y$ Bernoulli($p$).
2. La moyenne des embeddings de tokens est invariante à l'ordre ; la somme des positions est constante.
3. Une translation propre à chaque prompt disparaît des différences ; une multiplication ne disparaît pas à beta fixé.

</details>

## Pour aller plus loin

- [RLPT-00b — Biais des préférences](RLPT-00b-Biais-Dataset-Preferences-Python.ipynb).
- [RLPT-00d — RewardTrainer](RLPT-00d-Reward-Trainer-TRL-Python.ipynb).
- [RLPT-01 — PPO-RLHF](RLPT-01-PPO-RLHF-Petit-Modele-Python.ipynb).
- Références : Bradley et Terry (1952), Christiano et al. (2017), Ouyang et al. (2022), voir [index RL](../README.md#références).

- [FT-04 — RLHF et DPO](../../GenAI/FineTuning/FT-04-RLHF-DPO-Python.ipynb) : relier les préférences au post-entraînement de modèles de langage.